<a href="https://colab.research.google.com/github/afngh/transformers/blob/main/transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from rich.progress import track
import urllib.request
import math

In [11]:
class Embedding(nn.Module):
  def __init__(self, dims: int=64, vocab_size: int=5000, dropout: float=0.3):
    super().__init__()

    self.embed = nn.Embedding(vocab_size, dims)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    return self.dropout(self.embed(x))

In [12]:
class PositionalEmbedding(nn.Module):
  def __init__(self, dims: int=64, vocab_size: int=5000):
    super().__init__()

    pe = torch.zeros(vocab_size, dims)

    position = torch.arange(0, vocab_size, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dims, 2).float() * (-math.log(10000.0) / dims))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    self.pe = pe.unsqueeze(0)

  def forward(self, x):
    x = x + self.pe[:, :x.size(1)]
    return x

In [13]:
class Attention(nn.Module):
  def __init__(self, dims: int=64, dropout: float=0.3, head: int=4):
    super().__init__()

    self.dims = dims
    self.head = head
    self.hdim = dims // head

    self.dropout = nn.Dropout(dropout)

    self.q = nn.Linear(dims, dims)
    self.k = nn.Linear(dims, dims)
    self.v = nn.Linear(dims, dims)

    self.projection = nn.Linear(dims, dims)

  def forward(self,x):

    B, S, D = x.size()

    q = self.q(x)
    k = self.k(x)
    v = self.v(x)

    q = q.view(B, S, self.head, self.hdim).transpose(1, 2)
    k = k.view(B, S, self.head, self.hdim).transpose(1, 2)
    v = v.view(B, S, self.head, self.hdim).transpose(1, 2)

    scores = torch.matmul(q,k.transpose(-1,-2) / self.hdim ** 0.5)

    attention_weights = torch.softmax(scores, dim=-1)
    attention_weights = self.dropout(attention_weights)

    x = torch.matmul(attention_weights, v)
    x = x.transpose(1, 2).contiguous().view(B, S, D)
    x = self.projection(x)

    return x

In [14]:
class PostAttention(nn.Module):
  def __init__(self, dims: int=64, dropout: float=0.3):
    super().__init__()

    self.norm1 = nn.LayerNorm(dims)
    self.norm2 = nn.LayerNorm(dims)

    self.fnn = nn.Sequential(
        nn.Linear(dims, dims*4),
        nn.ReLU(),
        nn.Linear(dims*4, dims)
    )

    self.drop1 = nn.Dropout(dropout)
    self.drop2 = nn.Dropout(dropout)

  def forward(self, x, attention):
    x = x + self.drop1(attention)
    x = self.norm1(x)

    f = self.fnn(x)

    x = x + self.drop2(f)
    x = self.norm2(x)

    return x

In [15]:
class Transformer(nn.Module):
  def __init__(self, dims: int=64, vocab_size: int=5000, dropout: float=0.3, head: int=4):
    super().__init__()

    self.output_layer = nn.Linear(dims, vocab_size)

  def forward(self, x):
    return self.output_layer(x)

In [16]:
class Model(nn.Module):
  def __init__(self, embedding, pos_embeddings, attention, transformer):
    super().__init__()

    self.embedding = embedding
    self.pos_embeddings = pos_embeddings
    self.attention = attention
    self.transformer = transformer
  def forward(self, x):
    word_embedding = self.embedding(x)
    input = self.pos_embeddings(word_embedding)
    attention_output = self.attention(input)
    encoder_data = self.transformer(input, attention_output)

    last_vector = encoder_data[:,-1:,:].squeeze(0)
    logits = self.transformer(last_vector)
    return logits

In [17]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "text.txt")
with open("text.txt") as f:
    text = f.read(5000)

words = text.split()

vocab = sorted(list(set(words)))

wti = {word: i for i, word in enumerate(vocab)}
itw = {i: word for i, word in enumerate(vocab)}

seq = 5

X = []
y = []

for i in range(len(words)-seq):
  X.append([wti[w] for w in words[i:i+seq]])
  y.append([wti[words[i+seq]]])

X = torch.tensor(X)
y = torch.tensor(y)

x_test = torch.tensor([[21,  15,  10, 450]])
y_test = torch.tensor([[332]])

X

tensor([[ 21,  15,  10, 450, 332],
        [ 15,  10, 450, 332,  94],
        [ 10, 450, 332,  94, 201],
        ...,
        [ 54, 254, 219, 403, 443],
        [254, 219, 403, 443, 185],
        [219, 403, 443, 185,  54]])

In [18]:
embeddings = Embedding(vocab_size=len(vocab))
pos_embeddings = PositionalEmbedding(vocab_size=len(vocab))
attention = Attention()

transformer = Transformer(vocab_size=len(vocab))

word_embedding = embeddings(x_test)
input = pos_embeddings(word_embedding)
# print(input.shape)
encoder_output = PostAttention(dropout=0.3)

attention_output = attention(input)
encoder_data = encoder_output(input, attention_output)

last_vector = encoder_data[:,-1:,:].squeeze(0)

logits = transformer(last_vector)

model = Model(embeddings, pos_embeddings, attention, transformer)